<a href="https://colab.research.google.com/github/JpCurada/scheduled-qat-for-slm/blob/main/notebooks/01_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 1 — FP32 Baseline (Kaggle GPU)

**Goal:** establish ground-truth FP32 numbers for SmolLM2-1.7B that every later quantization experiment is measured against.

**This notebook produces:**
1. `results/baseline/baseline_results.json` — FP32 perplexity on WikiText-103 test, plus optional lm-eval scores.
2. `results/baseline/fp32_logits.pt` — saved FP32 output logits used as the reference distribution for KL-divergence in notebooks 02–05.

**Important:** when this notebook finishes, click **Save Version → Save & Run All**. After it commits, go to the version's *Output* tab and click **Add as Dataset** so notebooks 02–05 can mount the FP32 logits without re-running this notebook.

**Estimated time on a T4 GPU:** ~25 min (perplexity + logit save). Add ~3 h if `RUN_BENCHMARKS=True`.

## Troubleshooting — "Oops something went wrong"

That's a generic Kaggle frontend message; it doesn't say what crashed. Check these in order:

1. **GPU on?** Right sidebar → *Accelerator* → must be `GPU T4 x2` or `GPU P100`. CPU sessions will OOM trying to load a 1.7 B model.
2. **Internet on?** Right sidebar → *Internet* → must be `On`. Required for `git clone`, `pip install`, and the model download.
3. **Disk quota.** `/kaggle/working/` is capped at ~20 GB. The FP32 model (6.5 GB) + FP32 logits (1.6 GB) + pip cache fits with margin, but if you re-run multiple times without `Factory reset`, leftover state can fill it. Run `!df -h /kaggle/working` after the install cell to check.
4. **Kernel timeout / OOM during the FP32 forward.** SmolLM2-1.7B in FP32 is 6.5 GB; on a T4 (16 GB VRAM) we use `EVAL_BATCH = 1` to leave headroom. If you still OOM, drop `EVAL_SEQ_LEN` from 2048 to 1024.
5. **Frontend disconnect.** If the kernel survived but the browser dropped, *Save Version → Save & Run All* runs the notebook server-side and shows the real error in the version log.

## 1. Setup — clone repo, install package, GPU check

In [1]:
import os, sys, shutil, subprocess

In [22]:
 os.getcwd()

'/content/scheduled-qat-for-slm'

In [ ]:
/content/scheduled-qat-for-slm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [3]:
# === Kaggle setup ===
# Two options to bring in the source code:
#   A) git clone (default; needs Internet enabled in Kaggle settings)
#   B) mount the repo as a Kaggle Dataset and copy it (offline-friendly)
#
# Edit GITHUB_URL to your fork if needed.
WORKING_DIR  = "/content"
REPO_DIR     = f"{WORKING_DIR}/scheduled-qat-for-slm"
GITHUB_URL   = "https://github.com/JpCurada/scheduled-qat-for-slm.git"
REPO_DATASET = "/content/input/sqat-repo"   # only used if option B

if not os.path.exists(REPO_DIR):
    if os.path.exists(REPO_DATASET):
        print(f"Copying repo from Kaggle dataset: {REPO_DATASET}")
        shutil.copytree(REPO_DATASET, REPO_DIR)
    else:
        print(f"Cloning repo from: {GITHUB_URL}")
        subprocess.run(["git", "clone", "--depth", "1", GITHUB_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print("Repo:", REPO_DIR)
print("CWD :", os.getcwd())

Cloning repo from: https://github.com/JpCurada/scheduled-qat-for-slm.git
Repo: /content/scheduled-qat-for-slm
CWD : /content/scheduled-qat-for-slm


In [4]:
# Editable install — registers `src` as an importable package and installs deps.
# `-q` keeps Kaggle output clean; remove if you need to debug install issues.
!pip install -q -e ".[eval,viz]"

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.4/56.4 kB 3.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 79.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.1/91.1 kB 9.4 MB/s eta 0:00:00
  Building editable for scheduled-qat-slm (pyproject.toml) ... done


In [5]:
# Quick disk + GPU sanity check after install. Bail early with a clear error
# instead of letting OOM crash a later cell.
!df -h /content/ | tail -1
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv 2>/dev/null || echo "no GPU"

overlay         113G   44G   70G  39% /
name, memory.total [MiB], memory.free [MiB]
Tesla T4, 15360 MiB, 14913 MiB


In [6]:
import torch

torch.manual_seed(42)

print(f"torch    : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU      : {props.name}")
    print(f"VRAM     : {props.total_memory / 1e9:.1f} GB")
    print(f"Compute  : {props.major}.{props.minor}")
else:
    print("WARNING — running on CPU. FP32 SmolLM2-1.7B will be extremely slow.")

torch    : 2.10.0+cu128
CUDA     : True
GPU      : Tesla T4
VRAM     : 15.6 GB
Compute  : 7.5


## 2. Configuration

All Kaggle-specific knobs in one cell. Edit and re-run only this cell to change behaviour without re-running the setup.

In [7]:
from pathlib import Path

MODEL_NAME      = "HuggingFaceTB/SmolLM2-1.7B"
CACHE_DIR       = f"{WORKING_DIR}/models/base"
OUTPUT_DIR      = f"{WORKING_DIR}/results/baseline"

# Perplexity is reported at the standard SmolLM2 max context (2048).
# T4 has 16 GB VRAM; SmolLM2 in FP32 = 6.5 GB. batch=1 leaves headroom for activations.
EVAL_SEQ_LEN    = 2048
EVAL_BATCH      = 1

# Logits saved for KL divergence in later notebooks. Use 512-tok seqs to match the
# reduced training seq_length we'll use in QAT notebooks.
LOGIT_SEQ_LEN   = 512
LOGIT_SAMPLES   = 32         # 32 × 512 × 49152 × 2B fp16 ≈ 1.6 GB

# lm-evaluation-harness is slow (~3 h on T4 for full suite). Off by default.
RUN_BENCHMARKS  = False
BENCHMARK_TASKS = ["hellaswag", "arc_challenge", "piqa"]   # cheap subset

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
Path(CACHE_DIR).mkdir(parents=True, exist_ok=True)

print(f"Model      : {MODEL_NAME}")
print(f"Cache dir  : {CACHE_DIR}")
print(f"Output dir : {OUTPUT_DIR}")

Model      : HuggingFaceTB/SmolLM2-1.7B
Cache dir  : /content/models/base
Output dir : /content/results/baseline


## 3. Download SmolLM2-1.7B

First call downloads ~6.5 GB of FP32 weights into `CACHE_DIR`. Subsequent runs read from cache.

In [8]:
from src.training.baseline import download_model

download_model(model_name=MODEL_NAME, cache_dir=CACHE_DIR)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/635 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/831 [00:00<?, ?B/s]

## 4. FP32 perplexity + saved logits

`run_baseline()` does the full Phase 1 pass:
1. Load FP32 model on GPU.
2. Compute perplexity on the WikiText-103 **test** split (`seq_length=2048`).
3. Save logits from the WikiText-103 **train** split (`seq_length=512`, 32 samples) for KL divergence.

Step 3 produces the artifact every later notebook needs.

In [9]:
import gc
from src.training.baseline import run_baseline

# IMPORTANT: save_logits=False here. We save the logit file in the next cell at
# the smaller LOGIT_SEQ_LEN (512). Saving them inside run_baseline at 2048-len
# would burn ~6.4 GB of disk for data we'd immediately discard.
results = run_baseline(
    model_name=MODEL_NAME,
    cache_dir=CACHE_DIR,
    output_dir=OUTPUT_DIR,
    device_str="cuda",
    seq_length=EVAL_SEQ_LEN,
    eval_batch_size=EVAL_BATCH,
    save_logits=False,
    run_benchmarks=False,
)

print("\nBaseline summary:")
for k, v in results.items():
    print(f"  {k:20s}  {v}")

# Free the FP32 model loaded inside run_baseline before we load it again for logits.
gc.collect()
import torch as _t; _t.cuda.empty_cache()

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

README.md: 0.00B [00:00, ?B/s]

wikitext-103-raw-v1/test-00000-of-00001.(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-103-raw-v1/train-00000-of-00002(…):   0%|          | 0.00/157M [00:00<?, ?B/s]

...

Filtering empty lines (test):   0%|          | 0/4358 [00:00<?, ? examples/s]

Tokenizing test:   0%|          | 0/2891 [00:00<?, ? examples/s]

Chunking to 2048-token sequences (test):   0%|          | 0/2891 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()



Baseline summary:
  model                 HuggingFaceTB/SmolLM2-1.7B
  perplexity            8.206789874836801


### Save FP32 logits at the QAT training length (512)

This is the critical artifact every later notebook needs. We re-load the FP32 model fresh (memory was freed at the end of the perplexity cell) and run it on 32 sequences from the WikiText-103 train split, saving the output logits as float16. Total file size: ~1.6 GB.

KL divergence in notebooks 02–05 reads this file and re-runs the *quantized* model on the saved `input_ids`, then compares the two distributions.

In [10]:
import torch
from torch.utils.data import DataLoader
from transformers import AutoModelForCausalLM
from src.utils.data_loader import _load_and_chunk, get_tokenizer
from src.utils.metrics import save_fp32_logits

device = torch.device("cuda")
tokenizer = get_tokenizer(MODEL_NAME, CACHE_DIR)
ds = _load_and_chunk("wikitext-103-raw-v1", "train", tokenizer,
                     seq_length=LOGIT_SEQ_LEN, num_samples=LOGIT_SAMPLES)
loader = DataLoader(ds, batch_size=2, shuffle=False, num_workers=0, pin_memory=True)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, cache_dir=CACHE_DIR, torch_dtype=torch.float32
).to(device).eval()

logits_path = Path(OUTPUT_DIR) / "fp32_logits.pt"
save_fp32_logits(model, loader, logits_path, device, num_samples=LOGIT_SAMPLES)

size_gb = logits_path.stat().st_size / 1e9
print(f"Saved {LOGIT_SAMPLES} logit sequences @ seq_len={LOGIT_SEQ_LEN} "
      f"-> {logits_path} ({size_gb:.2f} GB)")

results["logits_path"] = str(logits_path)
results["logits_sequences"] = LOGIT_SAMPLES
results["logits_seq_length"] = LOGIT_SEQ_LEN
results["logits_size_gb"] = round(size_gb, 3)

del model, loader, ds
import gc; gc.collect()
torch.cuda.empty_cache()

Filtering empty lines (train):   0%|          | 0/1801350 [00:00<?, ? examples/s]

Tokenizing train:   0%|          | 0/1165029 [00:00<?, ? examples/s]

Chunking to 512-token sequences (train):   0%|          | 0/1165029 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

Saved 32 logit sequences @ seq_len=512 -> /content/results/baseline/fp32_logits.pt (1.61 GB)


## 5. (Optional) lm-evaluation-harness benchmarks

Set `RUN_BENCHMARKS=True` in the configuration cell to run this. Full suite (MMLU + HellaSwag + ARC + PIQA + GSM8K) takes ~3 h on a T4. The default `BENCHMARK_TASKS` list omits MMLU and GSM8K to keep it under an hour.

Skip this section if you're just establishing FP32 perplexity for the QAT comparisons — the FP32 benchmark numbers can be filled in later from the published SmolLM2 paper or a separate evaluation run.

In [15]:
RUN_BENCHMARKS=True

In [16]:
if RUN_BENCHMARKS:
    from src.utils.metrics import run_lm_eval
    bench = run_lm_eval(
        model_path=MODEL_NAME,
        output_path=Path(OUTPUT_DIR) / "lm_eval",
        tasks=BENCHMARK_TASKS,
        batch_size=8,
        device="cuda",
    )
    results["lm_eval"] = bench
    for task, r in bench.items():
        acc, err = r.get("acc"), r.get("acc_stderr")
        acc_s = f"{acc:.4f}" if acc is not None else "   n/a"
        err_s = f"{err:.4f}" if err is not None else "   n/a"
        print(f"  {task:20s}  acc={acc_s}  ±{err_s}")
else:
    print("Skipped (RUN_BENCHMARKS=False).")

config.json:   0%|          | 0.00/635 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/831 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.42G [00:00<?, ?B/s]

...

Generating train split:   0%|          | 0/16113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1838 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3084 [00:00<?, ? examples/s]

Running loglikelihood requests: 100%|██████████| 48531/48531 [1:32:10<00:00,  8.78it/s]


  arc_challenge         acc=0.4403  ±0.0145
  hellaswag             acc=0.5348  ±0.0050
  piqa                  acc=0.7682  ±0.0098


## 6. Persist final results

In [17]:
results

{'model': 'HuggingFaceTB/SmolLM2-1.7B',
 'perplexity': 8.206789874836801,
 'logits_path': '/content/results/baseline/fp32_logits.pt',
 'logits_sequences': 32,
 'logits_seq_length': 512,
 'logits_size_gb': 1.611,
 'lm_eval': {'arc_challenge': {'acc': 0.4402730375426621,
   'acc_stderr': 0.014506769524804264,
   'raw': {'alias': 'arc_challenge',
    'acc,none': 0.4402730375426621,
    'acc_stderr,none': 0.014506769524804264,
    'acc_norm,none': 0.47013651877133106,
    'acc_norm_stderr,none': 0.014585305840007114}},
  'hellaswag': {'acc': 0.5347540330611432,
   'acc_stderr': 0.00497771307389946,
   'raw': {'alias': 'hellaswag',
    'acc,none': 0.5347540330611432,
    'acc_stderr,none': 0.00497771307389946,
    'acc_norm,none': 0.713802031467835,
    'acc_norm_stderr,none': 0.004510593395290068}},
  'piqa': {'acc': 0.7682263329706203,
   'acc_stderr': 0.009845143772794095,
   'raw': {'alias': 'piqa',
    'acc,none': 0.7682263329706203,
    'acc_stderr,none': 0.009845143772794095,
    'ac

In [18]:
import json

summary_path = Path(OUTPUT_DIR) / "baseline_results.json"
with summary_path.open("w") as f:
    json.dump(results, f, indent=2, default=str)

print(f"Wrote {summary_path}")
print(json.dumps(results, indent=2, default=str))

Wrote /content/results/baseline/baseline_results.json
{
  "model": "HuggingFaceTB/SmolLM2-1.7B",
  "perplexity": 8.206789874836801,
  "logits_path": "/content/results/baseline/fp32_logits.pt",
  "logits_sequences": 32,
  "logits_seq_length": 512,
  "logits_size_gb": 1.611,
  "lm_eval": {
    "arc_challenge": {
      "acc": 0.4402730375426621,
      "acc_stderr": 0.014506769524804264,
      "raw": {
        "alias": "arc_challenge",
        "acc,none": 0.4402730375426621,
        "acc_stderr,none": 0.014506769524804264,
        "acc_norm,none": 0.47013651877133106,
        "acc_norm_stderr,none": 0.014585305840007114
      }
    },
    "hellaswag": {
      "acc": 0.5347540330611432,
      "acc_stderr": 0.00497771307389946,
      "raw": {
        "alias": "hellaswag",
        "acc,none": 0.5347540330611432,
        "acc_stderr,none": 0.00497771307389946,
        "acc_norm,none": 0.713802031467835,
        "acc_norm_stderr,none": 0.004510593395290068
      }
    },
    "piqa": {
      "ac

In [12]:
# Sanity check: confirm the artifacts the next notebooks expect.
!ls -lh {OUTPUT_DIR}

total 1.6G
-rw-r--r-- 1 root root  220 Apr 29 02:09 baseline_results.json
-rw-r--r-- 1 root root 1.6G Apr 29 02:08 fp32_logits.pt


## Next steps

1. **Save Version** in Kaggle (top-right) → *Save & Run All*.
2. After commit, open the version's **Output** tab and click **Add as Dataset**. Name it e.g. `sqat-baseline`.
3. In notebooks 02–05, add this dataset as input — they mount it at `/kaggle/input/sqat-baseline/results/baseline/fp32_logits.pt`.

Proceed to `02_ptq.ipynb`.

In [19]:
# 1. Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Copy the two required artifacts
import shutil
from pathlib import Path

DRIVE_DIR = Path('/content/drive/MyDrive/sqat-baseline/results/baseline')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)

shutil.copy('/content/results/baseline/fp32_logits.pt',     DRIVE_DIR)
shutil.copy('/content/results/baseline/baseline_results.json', DRIVE_DIR)

!ls -lh {DRIVE_DIR}


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
total 1.6G
-rw------- 1 root root 1.3K Apr 29 03:56 baseline_results.json
-rw------- 1 root root 1.6G Apr 29 03:56 fp32_logits.pt
